In [15]:
import pandas as pd
from recbole.config import Config
from UniSRec.unisrec import UniSRec
from UniSRec.data.dataset import UniSRecDataset
import numpy as np
from recbole.data.interaction import Interaction
import torch
from pathlib import Path
import pandas as pd

In [16]:
cfg_dict = {
    # paths
    "data_path": "new_folder6",
    "plm_suffix": "feat1CLS",
    "plm_size": 384,

    # training objective
    "loss_type": "CE",
    "train_neg_sample_args": None,

    # fields
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "ITEM_LIST_FIELD": "item_id_list",
    
    "TIME_FIELD": "timestamp",
    "load_col": {
        # "inter": ["user_id", "item_id", "timestamp"]
        "inter": ["user_id", "item_id", "item_id_list"]
    },

    # sequential
    "MAX_ITEM_LIST_LENGTH": 50,
    "max_seq_length": 50,

    # model (SASRec / UniSRec)
    "hidden_size": 300,
    "inner_size": 256,
    "n_layers": 2,
    "n_heads": 2,
    "hidden_dropout_prob": 0.4,
    "attn_dropout_prob": 0.4,
    "layer_norm_eps": 1e-12,
    "hidden_act": "gelu",
    "initializer_range": 0.02,
    
    "train_stage": "transductive_ft",
    
    # MoE adaptor
    "n_exps": 8,
    "adaptor_layers": [384, 300],   # must match hidden_size
    "adaptor_dropout_prob": 0.4,
    "adaptor_noise": False,

    # eval
    "eval_args": {
        # "split": {"RS": [0.8, 0.15, 0.05]},
        # "split": {"TS": [0.8, 0.15, 0.05]},
        # "split": {"LS": [0.8, 0.15, 0.05]},
        "group_by": "user",
        "order": "TO",
        "mode": "full"
    }
}


In [17]:
cfg_dict.update({
    # optimization
    "learning_rate": 1e-3,          # UniSRec default is relatively high
    "lr_scheduler": "cosine",
    "weight_decay": 1e-5,
    "train_batch_size": 2048,
    "benchmark_filename": ["train", "valid", "test"],
    "loss_type": "CE",

    # training control
    "epochs": 300,
    "eval_step": 1,                 # evaluate every epoch
    "stopping_step": 30,             # early stopping patience
    "clip_grad_norm": {
        "max_norm": 1.0,
        "norm_type": 2
    },

    # logging
    "log_wandb": False,
    "show_progress": True,

    # sequence field wiring (CRITICAL)
    "ITEM_LIST_LENGTH_FIELD": "item_length",
    "LIST_SUFFIX": "_list",
    "MAX_ITEM_LIST_LENGTH": 50,


    "train_neg_sample_args": None,
    "alias_of_item_id": None,
    "device": "cuda",

    "topk": [10,50],
    "metrics": ["Recall", "NDCG"],
    "valid_metric": "Recall@50",
    "eval_batch_size": 2048,
    "temperature": 0.07,
})

# cfg_dict["pretrained_model_path"] = "./UniSRec-FHCKM-300.pth"
config = Config(model=UniSRec, dataset="All_Beauty", config_dict=cfg_dict)


In [18]:
print("loss_type:", config["loss_type"])
print("train_neg_sample_args:", config["train_neg_sample_args"])
print("device:", config["device"])
print("pretrained_model_path:", config["pretrained_model_path"])
print("metrics:", config["metrics"])
print("topk:", config["topk"])

loss_type: CE
train_neg_sample_args: {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
device: cuda
pretrained_model_path: None
metrics: ['Recall', 'NDCG']
topk: [10, 50]


In [19]:
from recbole.config import Config
from recbole.data import data_preparation
from UniSRec.data.dataset import UniSRecDataset
from UniSRec.unisrec import UniSRec
from recbole.trainer import Trainer

# UniSRecDataset._benchmark_presets = debug_benchmark_presets
dataset = UniSRecDataset(config)

# ✅ THIS creates real DataLoaders with batching
train_data, valid_data, test_data = data_preparation(config, dataset)

# sanity-check one *real* batch
batch = next(iter(train_data))
print("user_id", batch["user_id"].shape)
print("item_id_list", batch["item_id_list"].shape)
print("item_length", batch["item_length"].shape)

model = UniSRec(config, dataset).to(config["device"])

trainer = Trainer(config, model)
pretrained_loaded = any("pretrained" in k.lower() for k in trainer.saved_model_file)
print("Loaded pretrained model:", pretrained_loaded)
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.norm().item())
        break

/home/hersco/.conda/envs/RecSys/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value="", inplace=True)
/home/hersco/.conda/envs/RecSys/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we

user_id
item_id_list
item_id


/home/hersco/.conda/envs/RecSys/lib/python3.10/site-packages/recbole/data/dataset/sequential_dataset.py:166: FutureWarning: using <built-in function len> in Series.agg cannot aggregate and has been deprecated. Use Series.transform to keep behavior unchanged.
  ].agg(len)
28 Dec 16:16    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}]
28 Dec 16:16    INFO  [Evaluation]: eval_batch_size = [2048] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


user_id torch.Size([2048])
item_id_list torch.Size([2048, 50])
item_length torch.Size([2048])
Loaded pretrained model: False
item_embedding.weight 59.15056610107422


In [20]:
print("train_data length:", len(train_data))
print("valid_data length:", len(valid_data))
print("test_data length:", len(test_data))
# print(dataset.__dict__)
print(len(dataset.field2id_token["item_id"]))

train_data length: 24
valid_data length: 5
test_data length: 1
29174


In [21]:
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("\n".join(trainable[:10]))
print("Trainable params:", len(trainable))


item_embedding.weight
position_embedding.weight
trm_encoder.layer.0.multi_head_attention.query.weight
trm_encoder.layer.0.multi_head_attention.query.bias
trm_encoder.layer.0.multi_head_attention.key.weight
trm_encoder.layer.0.multi_head_attention.key.bias
trm_encoder.layer.0.multi_head_attention.value.weight
trm_encoder.layer.0.multi_head_attention.value.bias
trm_encoder.layer.0.multi_head_attention.dense.weight
trm_encoder.layer.0.multi_head_attention.dense.bias
Trainable params: 54


In [ ]:
from recbole.utils import init_logger

init_logger(config)
best_valid_score, best_valid_result = trainer.fit(train_data, valid_data, show_progress=False, verbose=True)

In [ ]:
print("===== TRAIN LOSS PER EPOCH =====")
for epoch, loss in trainer.train_loss_dict.items():
    print(f"Epoch {epoch}: loss = {loss:.6f}")

In [ ]:
print("\n===== BEST VALIDATION =====")
print("Best valid score:", best_valid_score)
for k, v in best_valid_result.items():
    print(f"{k}: {v}")


In [ ]:
import torch

ckpt_path = "saved/UniSRec-Dec-27-2025_22-04-07.pth"
ckpt = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["state_dict"], strict=True)
model.to(config["device"])
model.eval()


In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from recbole.data.interaction import Interaction

# ------------------------------
# 0) Make CUDA errors synchronous (so stacktraces point to the real line)
# ------------------------------
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ------------------------------
# 1) Load model weights safely
# ------------------------------
DEVICE = config["device"]
ckpt_path = "saved/UniSRec-Dec-28-2025_12-50-12.pth"
ckpt = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["state_dict"], strict=True)
model.to(DEVICE)
model.eval()

# ------------------------------
# 2) Fields & mappings (authoritative)
# ------------------------------
uid_field = dataset.uid_field
iid_field = dataset.iid_field
item_seq_field = model.ITEM_SEQ
item_len_field = model.ITEM_SEQ_LEN

token2id = dataset.field2token_id[iid_field]   # ASIN -> internal id
id2token = dataset.field2id_token[iid_field]   # internal id -> ASIN

PAD_ID = 0
MAX_LEN = config["MAX_ITEM_LIST_LENGTH"]
TOP_K = 10
BATCH = 256

# sanity: vocab/emb sizes match
n_items = len(id2token)
assert n_items == model.item_embedding.num_embeddings == model.plm_embedding.num_embeddings, \
    (n_items, model.item_embedding.num_embeddings, model.plm_embedding.num_embeddings)

# ------------------------------
# 3) Load test
# ------------------------------
test_df = pd.read_csv("All_Beauty.test.csv")
assert "id" in test_df.columns and "history" in test_df.columns
N = len(test_df)

# ------------------------------
# 4) Build sequences (PAD-left, right-aligned) + CRITICAL: len>=1
# ------------------------------
seqs = np.full((N, MAX_LEN), PAD_ID, dtype=np.int64)
lens = np.zeros(N, dtype=np.int64)

empty_hist_count = 0
unk_tok_count = 0

for i, row in test_df.iterrows():
    hist = str(row["history"]) if pd.notna(row["history"]) else ""
    toks = hist.split()

    ids = []
    for a in toks:
        if a in token2id:
            ids.append(int(token2id[a]))
        else:
            unk_tok_count += 1

    ids = ids[-MAX_LEN:]

    # 🔴 CRITICAL FIX: enforce length >= 1
    if len(ids) == 0:
        empty_hist_count += 1
        ids = [PAD_ID]

    lens[i] = len(ids)
    seqs[i, -len(ids):] = ids  # right-align

print(f"Empty histories fixed: {empty_hist_count} / {N}")
print(f"Unknown tokens skipped (not in vocab): {unk_tok_count}")

# hard bounds checks (must pass)
assert lens.min() >= 1
assert lens.max() <= MAX_LEN
assert seqs.min() >= 0
assert seqs.max() < n_items

# dummy user ids (some models require it)
uids = np.zeros(N, dtype=np.int64)

# ------------------------------
# 5) Batched inference + robust filtering
# ------------------------------
all_preds = []

with torch.no_grad():
    for start in tqdm(range(0, N, BATCH), desc="Inference"):
        end = min(N, start + BATCH)

        inter = Interaction({
            uid_field: torch.from_numpy(uids[start:end]).long(),
            item_seq_field: torch.from_numpy(seqs[start:end]).long(),
            item_len_field: torch.from_numpy(lens[start:end]).long(),
        }).to(DEVICE)

        scores = model.full_sort_predict(inter)  # (B, n_items)

        top_ids = torch.topk(scores, k=TOP_K + 100, dim=1).indices.cpu().numpy()

        for r in range(end - start):
            seen = set(seqs[start + r].tolist())
            recs = []
            for idx in top_ids[r]:
                idx = int(idx)
                if idx != PAD_ID and idx not in seen:
                    recs.append(id2token[idx])  # internal id -> ASIN
                if len(recs) == TOP_K:
                    break

            # fallback (should be rare)
            if len(recs) < TOP_K:
                for idx in top_ids[r]:
                    idx = int(idx)
                    if idx != PAD_ID and id2token[idx] not in recs:
                        recs.append(id2token[idx])
                    if len(recs) == TOP_K:
                        break

            assert len(recs) == TOP_K
            all_preds.append(recs)

# ------------------------------
# 6) Build submission
# ------------------------------
submission = pd.DataFrame({"id": test_df["id"].values})
for i in range(TOP_K):
    submission[f"rec{i+1}"] = [row[i] for row in all_preds]

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv ready")
print(submission.head())
